# KOVA Voice Studio - Google Colab GPU Worker

Select **Runtime > Change runtime type > GPU**, then use **Runtime > Run all**. This worker refuses CPU fallback. When KOVA opens this notebook, the final cell sends the temporary worker session directly back to the already-open desktop app; no URL or token is copied.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/khoinguyen59/kova-video-dubbing.git'
REPO_REF = 'main'  # Replace with a KOVA release tag for production sessions.
WORKSPACE = Path('/content/kova-video-dubbing')

def run(command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, env=env, check=True)

run(['nvidia-smi'])
run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.8.13'])
if not WORKSPACE.exists():
    run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(WORKSPACE)])
else:
    # Colab keeps /content between runs. Always reset to the notebook's
    # requested revision so an older worker cannot miss new pairing routes.
    run(['git', 'fetch', '--depth', '1', 'origin', REPO_REF], cwd=WORKSPACE)
    run(['git', 'checkout', '--force', REPO_REF], cwd=WORKSPACE)
    run(['git', 'reset', '--hard', 'FETCH_HEAD'], cwd=WORKSPACE)
VOICE_DIR = WORKSPACE / 'voice-studio'
if not VOICE_DIR.exists():
    raise RuntimeError(f'Voice Studio source is missing: {VOICE_DIR}')
run(['uv', 'python', 'install', '3.11'])
run(['uv', 'venv', '--python', '3.11', '.venv'], cwd=VOICE_DIR)
PYTHON = str(VOICE_DIR / '.venv' / 'bin' / 'python')
OMNIVOICE_DIR = Path('/content/OmniVoice')
if not OMNIVOICE_DIR.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/k2-fsa/OmniVoice.git', str(OMNIVOICE_DIR)])
run(['uv', 'pip', 'install', '--python', PYTHON, '--index-url', 'https://download.pytorch.org/whl/cu128', 'torch==2.8.0+cu128', 'torchaudio==2.8.0+cu128'])
run(['uv', 'pip', 'install', '--python', PYTHON, '-r', 'requirements-colab.txt', 'demucs==4.0.1'], cwd=VOICE_DIR)
run(['uv', 'pip', 'install', '--python', PYTHON, str(OMNIVOICE_DIR)])
run(['uv', 'pip', 'install', '--python', PYTHON, '-e', '.'], cwd=VOICE_DIR)

## KOVA API Gateway (translation and preset TTS)

KOVA Desktop uses the gateway for free translation models and Google/Edge preset TTS. Voice Studio itself remains the separate OmniVoice cloning worker, so this notebook never copies a gateway key into the repository or a voice profile. If a Colab process later starts KOVA itself, store the key in the Colab secret named `KOVA_API_GATEWAY_API_KEY`; the code reads it only from runtime memory.

In [ ]:
GATEWAY_BASE_URL = 'http://3.27.172.90/v1'
FREE_TRANSLATION_MODELS = (
    'oc/deepseek-v4-flash-free', 'oc/mimo-v2.5-free', 'oc/hy3-free',
    'oc/big-pickle', 'oc/nemotron-3-ultra-free', 'oc/north-mini-code-free',
)
try:
    from google.colab import userdata
    gateway_key = userdata.get('KOVA_API_GATEWAY_API_KEY')
except Exception:
    gateway_key = None
if gateway_key:
    os.environ['KOVA_API_GATEWAY_API_KEY'] = gateway_key
    print('KOVA API Gateway secret loaded into this Colab runtime only.')
else:
    print('Gateway secret not loaded. This is fine for the Voice Studio worker; add the Colab secret only when KOVA runs in this runtime.')
print('Gateway base URL:', GATEWAY_BASE_URL)
print('Free translation models:', ', '.join(FREE_TRANSLATION_MODELS))
print('Verified preset TTS models: google-tts/vi, google-tts/en, edge-tts/vi-VN-HoaiMyNeural, edge-tts/vi-VN-NamMinhNeural')

In [ ]:
# KOVA Voice Studio notebook revision: 2026-07-22.
# Fail before model download if Colab resolves an incompatible package.
# IMPORTANT: this is Python __version__ (two underscores), never Markdown **version**.
import json
CHECK = r'''
import json, torch, transformers
from transformers import HiggsAudioV2TokenizerModel
from omnivoice import OmniVoice
assert torch.cuda.is_available(), 'CUDA is unavailable; stop and select a GPU runtime.'
version = getattr(transformers, '__version__', '')
assert version == '5.3.0', version
print(json.dumps({'device': 'cuda', 'gpu': torch.cuda.get_device_name(0), 'transformers': version}))
'''
doctor = subprocess.run([PYTHON, '-c', CHECK], cwd=VOICE_DIR, text=True, capture_output=True)
print(doctor.stdout, end='')
if doctor.returncode:
    raise RuntimeError('Dependency doctor failed. Reopen the current KOVA notebook from GitHub, then Run all.\n--- stderr ---\n' + doctor.stderr[-6000:])
print('Dependency doctor passed. The worker loads the model only on the first synthesis request.')

In [ ]:
import json, os, secrets, signal, subprocess, time, urllib.request
from pathlib import Path

# Run all can be executed more than once in the same Colab runtime. Stop
# the previous KOVA worker/tunnel first so a stale token never occupies
# port 3920 and makes the new health check return 401.
for pid_path in (Path('/content/kova-voice-worker.pid'), Path('/content/kova-voice-tunnel.pid')):
    try:
        pid = int(pid_path.read_text().strip())
        os.killpg(os.getpgid(pid), signal.SIGTERM)
    except (FileNotFoundError, ProcessLookupError, ValueError):
        pass
    finally:
        pid_path.unlink(missing_ok=True)
# Older notebook versions did not write PID files, so release their fixed
# listener as well. This only touches port 3920 used by KOVA Voice Studio.
subprocess.run(['bash', '-lc', 'fuser -k 3920/tcp >/dev/null 2>&1 || true'], check=False)
time.sleep(1)

TOKEN = secrets.token_urlsafe(32)
PAIR_CODE = secrets.token_urlsafe(32)
ENV = os.environ.copy()
ENV.update({
    'KOVA_VOICE_API_TOKEN': TOKEN,
    'KOVA_VOICE_PAIR_CODE': PAIR_CODE,
    'KOVA_VOICE_REQUIRE_CUDA': '1',
    'KOVA_VOICE_SEPARATE_REFERENCE': '1',
    'KOVA_VOICE_DATA_DIR': '/content/kova-voice-data',
})
PAIR_WORKER = VOICE_DIR / 'kova_pair_worker.py'
PAIR_WORKER.write_text('''from kova_voice_studio.api import create_app

# Keep the published worker API as the single source of truth. The desktop
# now receives the session through its callback relay, so no extra route can
# shadow the worker's own health or compatibility routes.
app = create_app()
''', encoding='utf-8')
LOG = '/content/kova-voice-worker.log'
LOG_HANDLE = open(LOG, 'w')
worker = subprocess.Popen([PYTHON, '-m', 'uvicorn', 'kova_pair_worker:app', '--host', '127.0.0.1', '--port', '3920'], cwd=VOICE_DIR, env=ENV, stdout=LOG_HANDLE, stderr=subprocess.STDOUT, start_new_session=True)
Path('/content/kova-voice-worker.pid').write_text(str(worker.pid))
for _ in range(45):
    try:
        request = urllib.request.Request('http://127.0.0.1:3920/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(request, timeout=3) as response:
            health = json.load(response)
        if health.get('ready') and health.get('device') == 'cuda':
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Voice worker did not become CUDA-ready. Log tail:\n' + Path(LOG).read_text(errors='replace')[-4000:])

run(['wget', '-q', '-O', '/content/cloudflared.deb', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'])
run(['dpkg', '-i', '/content/cloudflared.deb'])
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:3920', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
Path('/content/kova-voice-tunnel.pid').write_text(str(tunnel.pid))
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    marker = 'https://'
    if marker in line and 'trycloudflare.com' in line:
        public_url = line[line.find(marker):].split()[0]
        break
if not public_url:
    raise RuntimeError('Cloudflare tunnel URL was not found.')
print('\nKOVA_VOICE_URL=' + public_url)
from IPython.display import HTML, Javascript, display
display(HTML('<div style="font-family:Arial,sans-serif;padding:12px 16px;border-radius:8px;background:#eef6ff;color:#123;"><strong id="kova-pairing-status">KOVA đang gửi kết nối về desktop...</strong><br><span id="kova-pairing-detail">Không cần mở EXE lần hai, sao chép URL hoặc token.</span><br><button id="kova-pairing-retry" style="margin-top:8px;padding:10px 14px;border:0;border-radius:7px;background:#2f80ed;color:#fff;font-weight:700;cursor:pointer">Gửi lại kết nối</button></div>'))
relay_payload = json.dumps({'worker_url': public_url, 'token': TOKEN})
relay_script = """(() => { const status = document.getElementById('kova-pairing-status'); const detail = document.getElementById('kova-pairing-detail'); const retry = document.getElementById('kova-pairing-retry'); const query = new URLSearchParams(window.location.search); const callback = query.get('kova_callback'); const nonce = query.get('kova_nonce'); const send = async () => { if (!callback || !nonce) { status.textContent = 'Không thấy kênh ghép nối từ KOVA.'; detail.textContent = 'Hãy mở lại notebook bằng nút trong KOVA Voice Studio, sau đó Run all.'; return; } status.textContent = 'KOVA đang gửi kết nối về desktop...'; detail.textContent = 'Đợi vài giây rồi quay lại KOVA bấm Kiểm tra kết nối.'; try { const response = await fetch(callback + '/pair?nonce=' + encodeURIComponent(nonce), {method: 'POST', headers: {'Content-Type': 'application/json'}, body: __PAYLOAD__}); if (!response.ok) throw new Error(await response.text()); status.textContent = 'Đã gửi kết nối về KOVA.'; detail.textContent = 'Quay lại app đang mở và bấm Kiểm tra kết nối; không cần mở EXE khác.'; } catch (error) { status.textContent = 'Chưa gửi được kết nối.'; detail.textContent = 'Kênh ghép nối hết hạn hoặc bị chặn: ' + String(error) + '. Bấm Gửi lại kết nối hoặc mở Colab lại từ KOVA.'; } }; if (retry) retry.addEventListener('click', () => void send()); window.setTimeout(() => void send(), 300); })();""".replace('__PAYLOAD__', relay_payload)
display(Javascript(relay_script))
print('DEVICE=cuda')
print('KOVA tự gửi URL và token phiên về desktop qua kênh ghép nối riêng sau khi cell này chạy; không cần sao chép hoặc mở EXE lần hai.')